## 1. Extract all students that study Online, have an attendance_rate smaller than 50,
## and have more than 5 missed_deadlines. Store them in a dataframe called at_risk.

In [2]:
# import the libraries.
import pandas as pd
import numpy as np

In [3]:
# read the csv file and 5 rows of our data.
Student_df = pd.read_csv('student_dropout_risk_2026.csv.gz')
Student_df.head()


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


In [4]:
# get access to all of the columns.
Student_df.columns.tolist()

['person_id',
 'age',
 'gender',
 'nationality',
 'first_generation',
 'disability_status',
 'family_income_level',
 'scholarship_holder',
 'tuition_up_to_date',
 'part_time_job',
 'weekly_work_hours',
 'course_type',
 'study_mode',
 'semester',
 'international_student',
 'admission_grade',
 'previous_qualification_grade',
 'units_enrolled',
 'units_passed',
 'units_failed',
 'gpa_semester_1',
 'gpa_semester_2',
 'attendance_rate',
 'assignments_submitted_rate',
 'library_visits_per_month',
 'lms_logins_per_week',
 'extracurricular_activities',
 'missed_deadlines',
 'stress_level',
 'mental_health_support',
 'satisfaction_score',
 'commute_time_mins',
 'sleep_hours',
 'dropout']

In [5]:
at_risk = Student_df[
    (Student_df['study_mode'] == 'Online') &
    (Student_df['attendance_rate'] < 50) &
    (Student_df['missed_deadlines'] > 5)
]
at_risk

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,2,6.6,0,7,7,0,5,11,5.4,1
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,3,7.8,0,17,5,0,4,6,8.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96590,96591,23,Female,USA,1,0,Low,1,1,1,...,4,9.6,0,10,8,0,8,0,7.9,0
96648,96649,26,Female,USA,1,0,Low,1,1,1,...,2,12.4,0,8,9,1,6,10,6.5,1
97775,97776,19,Male,Germany,0,0,Low,1,0,1,...,6,7.5,1,8,6,1,10,19,8.8,1
99696,99697,26,Female,USA,1,0,Middle,1,0,1,...,2,14.3,1,16,9,1,8,12,7.8,1


## 2. Then compute, for each "course_type", how many of those students there are and what fraction of them has "dropped out". Store the result in a dataframe called "at_risk_by_course", with columns "students" and "dropout_rate", sorted by decreasing "dropout_rate".

In [7]:
at_risk_by_course = (
    at_risk.groupby('course_type').agg(students=('person_id', 'count'), 
    dropout_rate=('dropout', 'mean')).sort_values('dropout_rate', ascending=False)
)
at_risk_by_course

,students,dropout_rate
course_type,,
Law & Social Sciences,29,0.896552
Business,47,0.765957
STEM,47,0.765957
Arts & Humanities,40,0.750000
Health Sciences,34,0.647059


# 3. Add to the dataframe the following columns:

## pass_rate: the ratio between units_passed and units_enrolled;
## gpa_change: the difference between gpa_semester_2 and gpa_semester_1;
## gpa_trend: a string that is "improving" if gpa_change is larger than 0.5, "declining" if gpa_change is smaller than -0.5, and "stable" otherwise.

In [9]:
Student_df ['pass_rate'] = (Student_df['units_passed'] / Student_df['units_enrolled'])

Student_df ['gpa_change'] = (Student_df['gpa_semester_2'] - Student_df['gpa_semester_1'])

conditions = [
    Student_df['gpa_change'] > 0.5,
    Student_df['gpa_change'] < -0.5
]
choices = [
    'improving',
    'declining'
]

Student_df['gpa_trend'] = np.select(
    conditions, choices, default='stable'
)


Student_df[['units_passed', 'units_enrolled','pass_rate', 'gpa_change', 'gpa_trend']].head()

,units_passed,units_enrolled,pass_rate,gpa_change,gpa_trend
0,11,11,1.000000,-0.89,declining
1,11,14,0.785714,-0.03,stable
2,14,17,0.823529,-0.70,declining
3,11,13,0.846154,-0.41,stable
4,7,8,0.875000,-1.05,declining


## 4. Verify that, for every student, units_passed + units_failed equals units_enrolled; if not, report the rows violating this condition.



In [11]:
condition = (
    Student_df['units_passed'] + Student_df['units_failed']
    != Student_df['units_enrolled']
)
violations = Student_df.loc[condition]

violations


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend


## 5. Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop.

In [13]:
dropout_by_trend = (
    Student_df.groupby('gpa_trend')
    .agg(dropout_rate=('dropout', 'mean'))
)
dropout_by_trend

,dropout_rate
gpa_trend,
declining,0.265086
improving,0.589158
stable,0.391231


In [14]:
largest_gpa_drop = (
         Student_df.sort_values('gpa_change', ascending = True)
).head(10)
largest_gpa_drop[['person_id', 'gpa_change']]

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
76705,76706,-2.23
23427,23428,-2.23


## 6. Partition the students into the following groups, according to weekly_work_hours: 0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes).


In [16]:
conditions =[
(Student_df['weekly_work_hours'] == 0) ,
(Student_df['weekly_work_hours'] >= 1) & (Student_df['weekly_work_hours'] <= 10) ,
(Student_df['weekly_work_hours'] >= 11) & (Student_df['weekly_work_hours'] <= 20) ,
(Student_df['weekly_work_hours'] >= 21) & (Student_df['weekly_work_hours'] <= 30) ,
(Student_df['weekly_work_hours'] > 30)
]
choices = [
    '0 hours',
    '1-10 hours',
    '11-20 hours',
    '21-30 hours',
    'more than 30 hours'
]
Student_df['work_hours_group'] = np.select(
    conditions, choices, default='None'
)
#Student_df[['person_id', 'weekly_work_hours', 'work_hours_group']].head()

Student_df['work_hours_group'].value_counts()

work_hours_group
0 hours               51808
11-20 hours           23923
21-30 hours           15673
1-10 hours             6750
more than 30 hours     1846
Name: count, dtype: int64

# 7.Build a risk score by summing:
##    2 points if the tuition is not up to date;
##    1 point if stress_level is at least 8;
##    1 point if sleep_hours is smaller than 5;
##    1 point if the student has a part-time job with more than 20 weekly work hours;
##    2 points if attendance_rate is smaller than 60;
##    1 point if missed_deadlines is larger than 3.
### For each value of the risk score, compute the number of students and the dropout rate.


In [18]:
Student_df['risk_score'] = (
    (Student_df['tuition_up_to_date'] == 0) * 2 +
    (Student_df['stress_level'] >= 8) * 1 +
    (Student_df['sleep_hours'] < 5) * 1 +
    ((Student_df['part_time_job'] == 1) &
     (Student_df['weekly_work_hours'] > 20)) * 1 +
    (Student_df['attendance_rate'] < 60) * 2 +
    (Student_df['missed_deadlines'] > 3) * 1
)
Student_df['risk_score'].value_counts()

risk_score
0    26955
1    24115
2    21984
3    14557
4     7613
5     3320
6     1149
7      267
8       40
Name: count, dtype: int64

In [19]:
Student_df[['person_id', 'tuition_up_to_date']].head(10)

,person_id,tuition_up_to_date
0,1,0
1,2,1
2,3,1
3,4,1
4,5,1
5,6,1
6,7,0
7,8,1
8,9,1
9,10,1


In [20]:
Student_df['tuition_up_to_date'].value_counts()

tuition_up_to_date
1    78486
0    21514
Name: count, dtype: int64

In [21]:
risk_drop_out_rate = Student_df.groupby('risk_score').agg(
    students = ('person_id', 'count') , dropout_rate = ('dropout','mean'))
risk_drop_out_rate

,students,dropout_rate
risk_score,,
0,26955,0.152847
1,24115,0.252084
2,21984,0.337200
3,14557,0.452703
4,7613,0.577433
5,3320,0.673494
6,1149,0.794604
7,267,0.827715
8,40,0.900000


# 8. Read the GDP per capita dataset published by Our World in Data directly from the URL

## https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true
## and store it in a dataframe called gdp. The column ny_gdp_pcap_pp_kd contains the GDP per capita.

## Clean the dataset, removing the rows that do not correspond to a single country (e.g. World, High-income countries). For each country keep only the row with the most recent year.

In [29]:
gdp = pd.read_csv('https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true')
gdp.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


In [31]:
gdp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7445 entries, 0 to 7444
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   entity             7445 non-null   object 
 1   code               7445 non-null   object 
 2   year               7445 non-null   int64  
 3   ny_gdp_pcap_pp_kd  7445 non-null   float64
 4   owid_region        6977 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 290.9+ KB


In [33]:
gdp['code'].isna().sum()

0

In [35]:
gdp['code'].notna().sum()

7445

In [37]:
gdp[gdp['code'].notna()]

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


In [39]:
gdp['entity'].unique().tolist()

['Afghanistan',
 'Albania',
 'Algeria',
 'Andorra',
 'Angola',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Congo',
 'Costa Rica',
 "Cote d'Ivoire",
 'Croatia',
 'Curacao',
 'Cyprus',
 'Czechia',
 'Democratic Republic of Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Asia and Pacific (WB)',
 'East Timor',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Equatorial Guinea',
 'Estonia',
 'Eswatini',
 'Ethiopia',
 'Europe and Central Asia (WB)',
 'European Union (27)',
 'Faroe Islands',
 'Fiji',
 'Finland',
 'France',
 'Gabon',
 

In [41]:
aggregates = [
    'World',
    'High-income countries',
    'Low-income countries',
    'Lower-middle-income countries',
    'Upper-middle-income countries',
    'European Union (27)',
    'East Asia and Pacific (WB)',
    'Europe and Central Asia (WB)',
    'Latin America and Caribbean (WB)',
    'Middle East, North Africa, Afghanistan and Pakistan (WB)',
    'North America (WB)',
    'South Asia (WB)',
    'Sub-Saharan Africa (WB)'
]

gdp_clean = gdp[~gdp['entity'].isin(aggregates)]

In [43]:
gdp_clean.loc[gdp_clean['owid_region'].isna(), 'entity'].unique()

array([], dtype=object)

In [45]:
gdp_latest = (
    gdp_clean.sort_values('year', ascending=False)
             .drop_duplicates(subset='entity', keep='first')
)
gdp_latest[['entity', 'year', 'ny_gdp_pcap_pp_kd']].head(10)

,entity,year,ny_gdp_pcap_pp_kd
7444,Zimbabwe,2025,5529.2330
4855,Nicaragua,2025,7933.4890
1336,Chile,2025,30863.4220
4639,Myanmar,2025,5137.1006
1480,Congo,2025,6194.0900
4675,Namibia,2025,10242.2840
6722,Togo,2025,3485.9820
4711,Nauru,2025,12134.8920
706,Benin,2025,4116.2153
4747,Nepal,2025,5276.9240


In [47]:
gdp[gdp['entity'] == 'Italy']

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
3226,Italy,ITA,1990,42656.746,Europe
3227,Italy,ITA,1991,43283.020,Europe
3228,Italy,ITA,1992,43614.484,Europe
3229,Italy,ITA,1993,43216.110,Europe
3230,Italy,ITA,1994,44136.707,Europe
3231,Italy,ITA,1995,45410.140,Europe
3232,Italy,ITA,1996,46008.240,Europe
3233,Italy,ITA,1997,46869.426,Europe
3234,Italy,ITA,1998,47701.400,Europe
3235,Italy,ITA,1999,48493.664,Europe


In [49]:
gdp_latest.shape

(199, 5)

In [51]:
gdp_latest['entity'].duplicated().sum()

0

In [53]:
gdp_latest['year'].nunique()  #4 the latest year is not the same for all countries: 4 different years

4

## 9. Join the GPD data with the students data. Compute the dropout rate for each continent.

In [92]:
print(gdp.columns)
print(Student_df.columns)

Index(['entity', 'code', 'year', 'ny_gdp_pcap_pp_kd', 'owid_region'], dtype='object')
Index(['person_id', 'age', 'gender', 'nationality', 'first_generation',
       'disability_status', 'family_income_level', 'scholarship_holder',
       'tuition_up_to_date', 'part_time_job', 'weekly_work_hours',
       'course_type', 'study_mode', 'semester', 'international_student',
       'admission_grade', 'previous_qualification_grade', 'units_enrolled',
       'units_passed', 'units_failed', 'gpa_semester_1', 'gpa_semester_2',
       'attendance_rate', 'assignments_submitted_rate',
       'library_visits_per_month', 'lms_logins_per_week',
       'extracurricular_activities', 'missed_deadlines', 'stress_level',
       'mental_health_support', 'satisfaction_score', 'commute_time_mins',
       'sleep_hours', 'dropout', 'pass_rate', 'gpa_change', 'gpa_trend',
       'work_hours_group', 'risk_score'],
      dtype='object')


In [94]:
print(gdp['owid_region'].unique())
print(Student_df['nationality'].unique())

['Asia' 'Europe' 'Africa' 'North America' 'South America' 'Oceania' nan]
['Brazil' 'United States' 'Canada' 'Indonesia' 'Philippines' 'Egypt'
 'India' 'China' 'United Kingdom' 'Mexico' 'Nigeria' 'South Korea'
 'Pakistan' 'Germany' 'Australia']


In [100]:
gdp[gdp['entity'].str.contains('United States|United Kingdom', case=False, na=False)][['entity', 'owid_region']].drop_duplicates()

,entity,owid_region
7064,United Kingdom,Europe
7100,United States,North America
7136,United States Virgin Islands,North America


In [102]:
Student_df['nationality'] = Student_df['nationality'].replace({
    'USA': 'United States',
    'UK': 'United Kingdom'
})

students_gdp = pd.merge(
    Student_df,
    gdp_latest,
    left_on='nationality',
    right_on='entity',
    how='left'
)

In [104]:
students_gdp[['nationality', 'entity', 'owid_region']].drop_duplicates()

,nationality,entity,owid_region
0,Brazil,Brazil,South America
1,United States,United States,North America
4,Canada,Canada,North America
5,Indonesia,Indonesia,Asia
7,Philippines,Philippines,Asia
9,Egypt,Egypt,Africa
13,India,India,Asia
17,China,China,Asia
22,United Kingdom,United Kingdom,Europe
28,Mexico,Mexico,North America


In [106]:
dropout_by_continent = students_gdp.groupby('owid_region')['dropout'].mean()
dropout_by_continent

owid_region
Africa           0.316721
Asia             0.322908
Europe           0.322762
North America    0.317679
Oceania          0.321981
South America    0.317620
Name: dropout, dtype: float64

In [125]:
(dropout_by_continent * 100).round(2)

owid_region
Africa           31.67
Asia             32.29
Europe           32.28
North America    31.77
Oceania          32.20
South America    31.76
Name: dropout, dtype: float64

In [129]:
students_gdp.groupby('owid_region')['person_id'].count()

owid_region
Africa            7906
Asia             36035
Europe           10122
North America    38095
Oceania            969
South America     6873
Name: person_id, dtype: int64

The dropout rate is approximately 32% across the continents, with only small differences between them.

Oceania has the fewest students in the dataset, with 969 students, while North America has the most, with 38,095 students.

# Use of LLMs/Generative AI

I used ChatGPT to clarify Pandas concepts, receive guidance on merging datasets and calculating dropout rates, and debug my code. I ran the code, checked the outputs, and reviewed the results.